# Layer 2 — Findings Summary

Flagging engine output. Ranks findings from the descriptive pass by
significance and narrative relevance.

**What this notebook produces:**
- Correlation flags (strong pairs, sorted by strength)
- Changepoint clusters (multiple metrics shifting near the same year)
- Divergence flags (metrics moving in opposite directions)
- Outlier regions (states/districts that break the pattern)
- Custom hypothesis test results (from config.yaml)

**Action required:** Review the ranked findings below and mark which ones
to pursue in Layer 3 deep-dives.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
from pipeline.config import load_config, get_data_processed_dir

PROJECT_NAME = "qol-immigration"  # UPDATE per project

config = load_config(PROJECT_NAME)
processed_dir = get_data_processed_dir(config)

parquet_files = sorted(processed_dir.glob("*.parquet"))
datasets = {pf.stem: pd.read_parquet(pf) for pf in parquet_files}
print(f"Loaded {len(datasets)} dataset(s)")

## Correlation Flags

In [ ]:
from pipeline.flagging import flag_correlations

CORRELATION_THRESHOLD = 0.5  # |r| above this gets flagged

for name, df in datasets.items():
    print(f"\n--- {name} ---")
    flags = flag_correlations(df, threshold=CORRELATION_THRESHOLD)
    display(flags)

## Changepoint Clusters

In [ ]:
from pipeline.flagging import flag_changepoint_clusters

TIME_COL = "year"  # UPDATE if different

for name, df in datasets.items():
    if TIME_COL not in df.columns:
        continue
    print(f"\n--- {name} ---")
    clusters = flag_changepoint_clusters(df, time_col=TIME_COL)
    display(clusters)

## Divergence Flags

In [ ]:
from pipeline.flagging import flag_divergences

for name, df in datasets.items():
    if TIME_COL not in df.columns:
        continue
    print(f"\n--- {name} ---")
    divergences = flag_divergences(df, time_col=TIME_COL, config=config)
    display(divergences)

## Outlier Regions

In [ ]:
from pipeline.flagging import flag_outlier_regions

GEO_COL = "state"  # UPDATE if different

for name, df in datasets.items():
    if GEO_COL not in df.columns:
        continue
    print(f"\n--- {name} ---")
    outliers = flag_outlier_regions(df, geo_col=GEO_COL)
    display(outliers)

## Custom Hypothesis Tests

In [ ]:
from pipeline.flagging import test_hypotheses

hypotheses = config.get("hypotheses", [])
if not hypotheses:
    print("No custom hypotheses defined in config.yaml.")
else:
    results = test_hypotheses(datasets, hypotheses, config)
    for r in results:
        print(f"\n--- {r['name']} ---")
        print(f"Pattern: {r['pattern']}")
        print(f"Result: {r['summary']}")
        if r.get("figure"):
            r["figure"].show()

## Ranked Findings Summary

All flags combined and ranked by strength. Review and mark which to
pursue in Layer 3.

In [ ]:
from pipeline.flagging import compile_findings_summary

summary = compile_findings_summary(datasets, config,
                                    correlation_threshold=CORRELATION_THRESHOLD,
                                    time_col=TIME_COL,
                                    geo_col=GEO_COL)
display(summary)